In [ ]:
from circuits import build_circuit
import numpy as np
import shadows as sh
import pandas as pd

def normalize_new_features(unnormalized_features):
    from sklearn.preprocessing import MinMaxScaler

    scaler = MinMaxScaler(feature_range=(0,1))

    normalized_features = scaler.fit_transform(unnormalized_features)
    return normalized_features
    

def run_pipeline(filename = None, ring_paulis = ['XY'], entanglement = 'ring', num_layers = 1, encoding_axis = ("rx","ry"), train_test_val = None, filename_save = None):

    """
    filename: filename including path to file, ending in .csv, containing relevant data
    ring_paulis: Defines ring of specified observables. Takes an list of type list["PP'"] where P and P' are pauli observables (either I, X, Y or Z)
    entanglement: Defines entanglement of the circuit. Pass 'full', 'ring', or 'linear'
    num_layers: Number of ansatz layers. Layers involve data reuploading
    encoding_axis: Encoding the data via x-rotations, y-rotations, z-rotations, or some combination
    train_test_val: Adds the appropriate string to the of the file containing the new features
    filename_save: specify the name and path to save the new features to. Othwewise, it gets saved to the default location in Quantathon2025/Data. This name must be csv
    """
    
    df = pd.read_csv(f'../Data/{filename}')
    df = df.drop(df.columns[-1], axis = 1)


    data = df.to_numpy()
    n = len(data[0])


    circuits = [build_circuit(x, encoding_axes=encoding_axis, entanglement=entanglement, gate="cx", num_layers=num_layers)
                for x in data]
    
    paulis = sh.paulis_singles_xyz(n)

    for ring in ring_paulis:
        ring.upper()
        paulis += sh.paulis_ring_pairs(n, (ring[0], ring[1]))


    cfg = sh.ShadowConfig(T = 200, shots = 1000, seed = 123)

    new_features = sh.build_feature_matrix_from_circuits(circuits, paulis, cfg)
    
    normalized_features = normalize_new_features(new_features)

    df = pd.DataFrame(normalized_features)

    if filename_save:
        df.to_csv(filename_save, index = False)
    else:
        df.to_csv(f"../Data/shadow_features/{normalized_features.shape[1]}_features{ring_paulis[0]}{ring_paulis[1]}_{train_test_val}_QuantumLayers1.csv", index = False)


In [ ]:

run_pipeline(filename = '../../Data/normalized/X_val_scaled.csv', entanglement = 'full', train_test_val = 'val', num_layers = 1, filename_save= './temp2.csv', ring_paulis = ['XX', 'XY', 'YY'])